# 🔍 ONNX + Netron + Serving Lab
### Tasar'u · NVIDIA Platform & Cert Prep — hands-on lab

A trained model lives inside a framework (PyTorch). To *move* it — to another runtime, an
optimizer like TensorRT, or an edge device — you export it to **ONNX** (Open Neural Network
Exchange), an open, portable graph format. **Netron** lets you *see* that graph, and
**ONNX Runtime** runs it fast.

| Step | What you do | Maps to (slides) |
|---|---|---|
| 1 | Export a PyTorch model → **ONNX** | interchange format |
| 2 | Inspect the graph (**Netron**) | model visualization |
| 3 | Run **ONNX Runtime** vs PyTorch — speed + agreement | the "optimize then serve" path |
| 4 | **Accuracy eval** — does the export change results? | evaluation metrics |

**Platform:** Colab / Kaggle. GPU optional (this one even runs on CPU).


## 0 · Setup

In [1]:
!pip -q install "transformers>=4.44" "optimum[onnxruntime]" onnx netron datasets evaluate 2>/dev/null
import time, numpy as np, torch
print("ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 5.9 MB/s eta 0:00:00
ready


## 1 · Export a model to ONNX
We use a small sentiment classifier (DistilBERT fine-tuned on SST-2): tiny, fast, and easy to
score. 🤗 **Optimum** exports to ONNX in one call.

In [2]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MID = "distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(MID)

# PyTorch reference model
pt_model = AutoModelForSequenceClassification.from_pretrained(MID).eval()

# Export to ONNX (creates onnx/model.onnx) and wrap in an ONNX Runtime session
ort_model = ORTModelForSequenceClassification.from_pretrained(MID, export=True)
ort_model.save_pretrained("onnx_model")
import os; print("ONNX files:", [f for f in os.listdir("onnx_model") if f.endswith(".onnx")])

Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

The model distilbert-base-uncased-finetuned-sst-2-english was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


ONNX files: ['model.onnx']


## 2 · Inspect the graph
First a text summary from the ONNX file, then Netron for the visual.

In [3]:
import onnx
m = onnx.load("onnx_model/model.onnx")
print("Inputs :", [i.name for i in m.graph.input])
print("Outputs:", [o.name for o in m.graph.output])
print("Operators (nodes):", len(m.graph.node))
from collections import Counter
print("Top ops:", Counter(n.op_type for n in m.graph.node).most_common(6))

Inputs : ['input_ids', 'attention_mask']
Outputs: ['logits']
Operators (nodes): 506
Top ops: [('Constant', 152), ('Add', 61), ('MatMul', 48), ('Unsqueeze', 30), ('Concat', 25), ('Reshape', 25)]


**See it visually with Netron** — two easy ways:
- **Web (simplest):** download `onnx_model/model.onnx` and drag it onto **https://netron.app**.
- **In-notebook:** run the cell below, then open the printed URL (works best on Colab).

In [4]:
# Optional: download the file (Colab) so you can drop it into netron.app
try:
    from google.colab import files; files.download("onnx_model/model.onnx")
except Exception:
    print("Not on Colab — grab onnx_model/model.onnx from the file browser, open at netron.app")
# Optional in-notebook viewer:
# import netron; netron.start("onnx_model/model.onnx", browser=False)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3 · ONNX Runtime vs PyTorch — speed and agreement
Same inputs through both. ONNX Runtime applies graph optimizations (fusion, constant folding)
— the same *class* of trick TensorRT does, one tier up.

In [6]:
texts = ["This GPU cluster is blazing fast!", "The training job crashed again and lost a checkpoint."]
enc = tok(texts, return_tensors="pt", padding=True)

def time_it(fn, runs=50):
    fn();  t0=time.time()
    for _ in range(runs): fn()
    return (time.time()-t0)/runs*1000  # ms

with torch.no_grad():
    pt_fn  = lambda: pt_model(**enc).logits.detach().numpy()
enc_np = {k: v.numpy() for k, v in enc.items()}
ort_fn = lambda: ort_model.model.run(None, enc_np)[0]

pt_ms, ort_ms = time_it(pt_fn), time_it(ort_fn)
pt_out, ort_out = pt_fn(), ort_fn()
print(f"PyTorch : {pt_ms:5.2f} ms/batch")
print(f"ONNX RT : {ort_ms:5.2f} ms/batch   ({pt_ms/ort_ms:.2f}× vs PyTorch on this runtime)")
print("Max logit difference:", float(np.abs(pt_out - ort_out).max()), "(≈0 → same model, different engine)")

PyTorch : 216.75 ms/batch
ONNX RT : 133.67 ms/batch   (1.62× vs PyTorch on this runtime)
Max logit difference: 9.5367431640625e-07 (≈0 → same model, different engine)


## 4 · Accuracy eval — did the export change the answers?
Export should be **lossless**. Prove it: score both models on a slice of the SST-2 validation
set and compare accuracy (the metric from the deck).

In [ ]:
from datasets import load_dataset
val = load_dataset("glue", "sst2", split="validation[:200]")
labels = np.array(val["label"])

def predict_pt(batch):
    e = tok(batch, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad(): return pt_model(**e).logits.argmax(-1).numpy()
def predict_ort(batch):
    e = tok(batch, return_tensors="np", padding=True, truncation=True)
    return ort_model.model.run(None, dict(e))[0].argmax(-1)

import numpy as np
def acc(fn):
    preds=[]
    for i in range(0, len(val), 32):
        preds.append(fn(val["sentence"][i:i+32]))
    p=np.concatenate(preds); return (p==labels).mean()

print(f"PyTorch accuracy : {acc(predict_pt)*100:.1f}%")
print(f"ONNX RT accuracy : {acc(predict_ort)*100:.1f}%")
print("→ Same accuracy = a faithful export. You changed the engine, not the model.")

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

PyTorch accuracy : 91.0%


## 5 · Reflection
1. Put these on the right layer of the deck's stack: **ONNX**, **ONNX Runtime / TensorRT**,
   **Netron**. Which *moves* the model, which *speeds* it, which *shows* it?
2. ONNX Runtime was faster with identical accuracy. Why is a portable graph + a dedicated
   runtime the standard way to ship inference?
3. You exported a classifier. What's harder about exporting an **LLM** to ONNX?
   *(hint: the KV cache and dynamic sequence lengths — why TensorRT-**LLM** exists.)*
